# Lecția 01 - Introducere în agenții AI

Bine ați venit la prima lecție din cursul **Agenți AI pentru Începători**!

Un **agent AI** este un program care folosește un model lingvistic mare (LLM) ca motor de raționament și poate lua *acțiuni* în lumea reală — apelând API-uri, interogând baze de date sau rulând cod — pentru a îndeplini un scop în numele unui utilizator.

În acest caiet veți construi primul agent: un **Agent de Călătorii** care recomandă destinații de vacanță. Pe parcurs veți învăța cum să:

1. Vă conectați la Microsoft Foundry Agent Service folosind **Microsoft Agent Framework**.
2. Oferiți agentului un **instrument** — o funcție Python simplă pe care o poate apela.
3. Rulați agentul și examinați răspunsul acestuia.
4. Transmiteți răspunsul agentului token cu token.


## Configurare

Înainte de a rula acest notebook, asigură-te că ai:

1. **Un proiect Microsoft Foundry** cu un model de chat implementat (de ex. `gpt-4.1-mini`).
2. **Conectat cu Azure CLI** — rulează `az login` în terminalul tău.
3. **Setat variabilele de mediu necesare:**
   - `AZURE_AI_PROJECT_ENDPOINT` — punctul tău final al proiectului Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — numele modelului tău implementat.

Celula de mai jos instalează pachetele Python de care ai nevoie.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Crearea primului tău agent

Un agent are nevoie de două lucruri:

- **Instrucțiuni** care îi spun *cine este* și *cum să se comporte* (un prompt de sistem).
- **Unelte** — funcții Python decorate cu `@tool` pe care agentul le poate apela pentru a recupera informații sau pentru a efectua acțiuni.

Mai jos definim o unealtă simplă care returnează o listă de destinații populare de vacanță. Agentul va folosi această unealtă când un utilizator solicită recomandări de călătorie.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Răspunsuri în flux

Pentru o experiență mai interactivă, puteți **transmite în flux** răspunsul agentului. În loc să așteptați răspunsul complet, agentul oferă fragmente de text pe măsură ce sunt generate. Acest lucru este util mai ales în interfețele de chat unde doriți să afișați rezultatul în timp real.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Rezumat

În această lecție ai învățat să:

- **Creezi un furnizor** care se conectează la Microsoft Foundry Agent Service prin `FoundryChatClient`.
- **Definiți un instrument** folosind decoratorul `@tool` pentru ca agentul să poată apela funcțiile tale Python.
- **Rulezi agentul** cu un mesaj de la utilizator și să afișezi răspunsul acestuia.
- **Transmiți răspunsurile** pentru ieșire în timp real.

În următoarea lecție vom explora cadrele agentice mai în profunzime și vom învăța cum să oferim agenților instrumente mai puternice și capacități de raționare în mai mulți pași.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
